# 00 Synthetic Replay

This notebook runs the first simulator milestone: 10,000 synthetic blocks through Mechanism A, where bandwidth is priced as `calldata_bytes + bal_bytes` and state remains under the EIP-8037 `max(regular_gas_used, state_gas_used)` baseline.

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sim import generate_synthetic_blocks, load_config, replay
from sim.metrics import compute_metrics
from sim.plots import make_main_plots


## Load Config

Change `bandwidth.target_bytes`, `bandwidth.limit_bytes`, `bandwidth.update_fraction`, `bandwidth.reserve_mode`, `synthetic.correlation_calldata_bal`, `synthetic.bal_burst_probability`, or `synthetic.state_burst_probability` in `configs/synthetic_bandwidth_only.yaml`, then rerun the notebook.

In [ ]:
config = load_config(PROJECT_ROOT / "configs" / "synthetic_bandwidth_only.yaml")
config


## Generate Synthetic Blocks

The generator creates explicit normal, L2 calldata burst, BAL burst, state-growth burst, and correlated-stress regimes.

In [ ]:
blocks = generate_synthetic_blocks(config)
blocks.head()


In [ ]:
blocks["regime"].value_counts(normalize=True).rename("share").to_frame()


## Replay Mechanism A

In [ ]:
df = replay(blocks, config)
metrics = compute_metrics(df, config)
metrics.T


## Main Plots

The five first-pass plots are: bandwidth components, bandwidth base fee, shared execution/state base fee, bandwidth usage over limit, and state gas vs BAL bytes.

In [ ]:
figures = make_main_plots(df, config)


## Experiment 1: Calldata Only vs Calldata + BAL

Run the same synthetic blocks with `bal_bytes = 0`, then compare fee paths and metrics against the full bandwidth definition.

In [ ]:
calldata_only_blocks = blocks.assign(bal_bytes=0)
df_calldata_only = replay(calldata_only_blocks, config)

comparison = pd.concat(
    {
        "calldata_only": compute_metrics(df_calldata_only, config).iloc[0],
        "calldata_plus_bal": compute_metrics(df, config).iloc[0],
    },
    axis=1,
).T
comparison


In [ ]:
ax = df_calldata_only.plot(
    x="block_number",
    y="bandwidth_base_fee",
    label="calldata only",
    figsize=(11, 4),
    linewidth=1.0,
)
df.plot(
    x="block_number",
    y="bandwidth_base_fee",
    label="calldata + BAL",
    ax=ax,
    linewidth=1.0,
)
ax.set_title("Bandwidth base fee: calldata only vs calldata + BAL")
ax.set_xlabel("block")
ax.set_ylabel("fee units per byte");


## Example In-Notebook Override

Use `dataclasses.replace` for one-off runs without editing YAML.

In [ ]:
# example_config = replace(
#     config,
#     bandwidth=replace(config.bandwidth, target_bytes=600_000, limit_bytes=1_500_000),
#     synthetic=replace(config.synthetic, correlation_calldata_bal=0.6),
# )
# example_df = replay(generate_synthetic_blocks(example_config), example_config)
# compute_metrics(example_df, example_config).T
